# 단계 별 실행 값 확인

### 1. 추정 경계 청크 불러오기 및 1차 요약문 생성

In [ ]:
# 커널 재시작하지 않아도 수정된 코드 자동으로 반영되도록 설정
%load_ext autoreload
%autoreload 2

import json
import pandas as pd

import llm
from node.load_input import load_input
from node.segment_summary import select_candidates, summarize_segment, segment_summary_node

import tracing
print(tracing.tracing_status())    # "LangSmith 기록 켜짐 (프로젝트: REMEET)"


# ── 설정 ──
BOUNDARY_FILE = "../../dataset/Text_Boundaries/TS3003c_boundary_candidates.json"
TEST_SEGMENT_ID = 1          # 단건 테스트할 Segment 번호

print("모델:", llm.OPENAI_MODEL, "/ effort:", llm.LLM_REASONING_EFFORT, "/ 캐시:", llm.LLM_CACHE_DIR)

In [ ]:
# 입력 불러오기
state = load_input(BOUNDARY_FILE)
print(state["meeting_id"], state["metadata"])
pd.DataFrame([{
    "segment": s.segment_id,
    "핵심 범위": f"{s.utterance_start}~{s.utterance_end}",
    "오버랩 포함": f"{s.overlap_start}~{s.overlap_end}",
    "핵심 발화 수": len(s.utterances),
} for s in state["segments"]])


In [ ]:
# LLM에 보낼 프롬프트 미리보기
from prompt.segment_summary_prompt import SYSTEM_PROMPT, build_user_prompt

seg = state["segments"][TEST_SEGMENT_ID]
candidates = select_candidates(state["registry"], seg)
print(build_user_prompt(seg, candidates, state["metadata"]))


In [ ]:
# Segment 하나만 1차 요약 (LLM 1회 호출됨)
r = summarize_segment(seg, state["registry"], [], state["meeting_id"], state["metadata"])

print("── LLM 원본 출력 (ID 없음) ──")
print(r["llm_output"].model_dump_json(indent=2))
print("\n── 후처리 로그 ──")
print("\n".join(r["log"]) or "(없음)")


In [ ]:
# ID 부여 결과 표로 확인
def items_table(summaries):
    rows = []
    for s in summaries:
        for a in s.agendas:
            for section, item in a.iter_items():
                rows.append({"segment": s.segment_id, "agenda_id": a.agenda_id, "안건": a.agenda_title,
                             "섹션": section, "item_id": item.item_id,
                             "tag": getattr(item, "tag", ""), "subtopic": getattr(item, "subtopic", ""),
                             "text": item.text, "근거": item.evidence_utterance_indices})
    return pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 80)
items_table([r["summary"]])


In [ ]:
# 전체 Segment 3개만 활용한 1차 요약 (LLM 3회 호출)
state["segments"] = state["segments"][:3] # 앞 3개 Segment만
state.update(segment_summary_node(state))
items_table(state["segment_summaries"])

In [ ]:
# 전체 결과 확인
# 안건 Registry
display(pd.DataFrame([a.model_dump() for a in state["registry"].agendas]))

# 전체 요약 항목
items_table(state["segment_summaries"])


In [ ]:
# 근거 발화 원문 대조 (특정 item 확인)
ITEM_ID = "TS3003c_a1_d1"     # 확인할 항목

for s in state["segment_summaries"]:
    for a in s.agendas:
        item = a.find_item(ITEM_ID)
        if item:
            print(f"[{item.item_id}] {item.text}\n")
            for i in item.evidence_utterance_indices:
                print(f"  [{i}] {state['utterance_texts'][i]}")


### 2. 안건 기준 1차 병합

In [ ]:
from node.agenda_merge import group_by_agenda, agenda_merge_node

# 병합 실행 (LLM 호출 없음)
state.update(agenda_merge_node(state))

# 안건별 결과 확인 (셀 5의 items_table과 같은 형태)
pd.DataFrame([{"agenda_id": a.agenda_id, "안건": a.agenda_title, "섹션": sec, "item_id": it.item_id,
               "status": it.status, "tag": getattr(it, "tag", ""), "text": it.text,
               "근거": it.evidence_utterance_indices, "Segment": it.source_segment_ids}
              for a in state["agendas"] for sec, it in a.iter_items(include_deleted=True)])


In [ ]:
def agendas_to_json(state, include_deleted=True):
    # 저장할 JSON 구성 (include_deleted=False면 DELETED 항목 제외하고 확인)
    agendas = []
    for a in state["agendas"]:
        d = a.model_dump()
        if not include_deleted:
            for sec in ("discussion_background", "major_discussions", "decisions",
                        "unresolved_issues", "next_meeting_topics"):
                d[sec] = [it for it in d[sec] if it["status"] != "DELETED"]
        agendas.append(d)
    return {"meeting_id": state["meeting_id"], "metadata": state["metadata"], "agendas": agendas}

merged_json = agendas_to_json(state)
print(json.dumps(merged_json, ensure_ascii=False, indent=2))


In [ ]:
# 안건 list 예)
{
  "meeting_id": "TS3003c",
  "agendas": [
    {
      "agenda_id": "TS3003c_a1",
      "agenda_title": "콘셉트 설계",
      "evidence_range": [9, 84] # [9, 12], [9,59], [9,84]로 확장됨
    }
  ],
  "next_number": 2 # 다음에 새 안건이 생기면 붙일 번호
}


In [ ]:
# * 안건별 병합 청크 크기 확인 (LLM 호출 없음)
from node.content_validation import source_utterances
from prompt.content_validation_prompt import SYSTEM_PROMPT, build_user_prompt, format_utterances

try:
    import tiktoken
    enc = tiktoken.get_encoding("o200k_base")
    count_tokens = lambda s: len(enc.encode(s))
except ImportError:
    count_tokens = lambda s: None          # tiktoken이 없으면 토큰 수는 비워 둠

rows = []
for a in state["agendas"]:
    utts = source_utterances(a, state["segments"], state["utterance_texts"])
    idx = [i for i, _ in utts]
    user_prompt = build_user_prompt(a, utts)
    rows.append({
        "agenda_id": a.agenda_id,
        "title": a.agenda_title,
        "segments": a.source_segment_ids,
        "items": sum(1 for _, it in a.iter_items() if it.status == "ACTIVE"),
        "utterances": len(utts),
        "range": f"{idx[0]}~{idx[-1]}" if idx else "-",
        "gaps": format_utterances(utts).count("\n...\n"),          # 끊긴 구간 수
        "utt_chars": sum(len(t) for _, t in utts),
        "prompt_chars": len(SYSTEM_PROMPT) + len(user_prompt),
        "prompt_tokens": count_tokens(SYSTEM_PROMPT + user_prompt),
    })

size_df = pd.DataFrame(rows)
display(size_df)
print(f"안건 {len(size_df)}개 / 발화 합계 {size_df['utterances'].sum()} / 최대 프롬프트 {size_df['prompt_chars'].max():,}자")


### 3. ( 내용 검증 Agent + 재수정 ) 반복

In [ ]:
from node.content_validation import source_utterances, validate_agenda, content_validation_node

# 프롬프트 미리 보기 (LLM 호출 없음)
from prompt.content_validation_prompt import build_user_prompt
a = state["agendas"][0]
print(build_user_prompt(a, source_utterances(a, state["segments"], state["utterance_texts"])))

In [ ]:
# (2) 안건 하나만 검증 (LLM 1회)
r = validate_agenda(a, state["segments"], state["utterance_texts"], state["meeting_id"])
print(r["llm_output"].model_dump_json(indent=2))
print("\n".join(r["log"]) or "(코드 검사 변경 없음)")


In [ ]:
# (3) 전체 안건 1회차 검증 (안건 수만큼 LLM 호출) → 이슈 표
state.update(content_validation_node(state))
pd.DataFrame([i.model_dump() for i in state["content_issue_log"]])


In [ ]:
from node.content_revise import content_revise_node, revise_agenda

# 재수정 (이슈가 있는 안건만 LLM 호출)
if state.get("content_issues"):
    state.update(content_revise_node(state))
    print("재검증 대상:", state["recheck_item_ids"])
else:
    print("이슈 없음 → 재수정 생략")

In [ ]:
# 수정된 항목의 수정 전/후 비교
rows = []
for a in state["agendas"]:
    for sec, it in a.iter_items(include_deleted=True):
        for h in it.history:
            if h.stage == "content_revise":
                rows.append({"item_id": it.item_id, "status": it.status, "loop": h.loop,
                             "before": h.before_text, "after": it.text,
                             "tag": f"{h.before_tag} → {getattr(it, 'tag', None)}", "reason": h.reason})
pd.DataFrame(rows)


In [ ]:
# 남은 이슈가 없거나 최대 회차에 도달할 때까지 검증과 재수정 반복
from node.content_validation import route_after_content_validation
while state.get("content_issues"):
    state.update(content_revise_node(state))
    state.update(content_validation_node(state))
print("다음 단계:", route_after_content_validation(state))


### 4. ( 가독성 검증 Agent + 재수정 ) 반복

In [ ]:
from node.readability_validation import (evidence_utterances, validate_agenda as validate_readability,
                                         readability_validation_node, route_after_readability_validation)
from node.readability_revise import readability_revise_node
from prompt.readability_validation_prompt import build_user_prompt as build_readability_prompt

# (1) 프롬프트 미리 보기 (LLM 호출 없음)
a = state["agendas"][0]
print(build_readability_prompt(a, evidence_utterances(a, state["utterance_texts"])))


In [ ]:
# (2) 1회차 가독성 검증 → 이슈 표
state.update(readability_validation_node(state))
pd.DataFrame([i.model_dump(include={"issue_id", "target_item_id", "criterion", "current_text", "revised_text", "reason"})
              for i in state["readability_issues"]])


In [ ]:
# (3) 남은 이슈가 없거나 최대 회차에 도달할 때까지 수정과 재검증 반복
while state.get("readability_issues"):
    state.update(readability_revise_node(state))
    state.update(readability_validation_node(state))
print("다음 단계:", route_after_readability_validation(state))


In [ ]:
issue_df = pd.DataFrame([i.model_dump(include={"loop", "issue_id", "target_item_id", "criterion", "current_text",
                                               "reason", "revision_instruction", "applied", "revised_text"})
                         for i in state["readability_issue_log"]])
issue_df = issue_df[["loop", "issue_id", "target_item_id", "criterion", "applied",
                     "current_text", "revised_text", "reason", "revision_instruction"]]
print(issue_df.groupby("loop")["applied"].agg(["count", "sum"]).rename(columns={"count": "이슈", "sum": "반영"}))
display(issue_df)


In [ ]:
final_rows = []
for a in state["agendas"]:
    for sec, it in a.iter_items():
        revs = [h for h in it.history if h.stage == "readability_revise"]
        if revs:
            final_rows.append({"item_id": it.item_id, "수정 횟수": len(revs),
                               "처음 문장": revs[0].before_text, "최종 문장": it.text})
pd.DataFrame(final_rows)


### 5. (구조 검증 Agent + 재수정 ) 반복

In [ ]:
from node.structure_validation import structure_validation_node

state.update(structure_validation_node(state))
rep = state["structure_report"]
print(rep.validation_status, "/ 재검증", rep.retry_count, "회 / 수동 검토 필요:", rep.requires_manual_review)
display(pd.DataFrame([d.model_dump() for d in rep.error_details]))     # 오류가 없으면 빈 표
for q in state["quarantine"]:
    print("격리:", q["kind"], q.get("item_id") or q.get("agenda_id"), [e["error_type"] for e in q["errors"]])


In [ ]:
import json
from node.structure_validation import validate_structure_json

data = [a.model_dump(mode="json") for a in state["agendas"]]
data[0]["decisions"][0]["tag"] = "합의"                              # 일부러 오류 넣기
res = validate_structure_json(json.dumps(data, ensure_ascii=False), state["meeting_id"],
                              state["segments"], state["utterance_texts"], state["registry"])
pd.DataFrame([d.model_dump() for d in res["report"].error_details])


### 6. 최종 병합

#### FinalSummary Json 구조

```text
FinalSummary
├─ meeting_info                                   # 회의 정보
│   ├─ title / date / participants                # ← metadata의 "회의 대주제" / "일시" / "참여자"
│   └─ source_document                            # ← 원본 전사문 전문
│       ├─ utterance_count
│       ├─ utterances [{utterance_index, text}]   # (근거 발화 연결용)
│       └─ full_text
├─ agenda_index []                                # 안건 목차: number, agenda_id, agenda_title, 
│                                                 #           decision_count, unresolved_count
├─ meeting_results []                             # 회의 결과: section, tag, text, agenda_number, 
│                                                 #           item_id, evidence, review_required
│
└─ agenda_details []                              # 안건별 상세
    ├─ number / agenda_id / agenda_title
    ├─ discussion_background / decisions / unresolved_issues / next_meeting_topics []
    │    # 항목: item_id, tag, text, evidence_utterance_indices, review_required
    └─ major_discussions [] 소주제 묶음: number("1-1"), subtopic, items[]
```

In [ ]:
from node.render import render_node, render_meeting_info, render_agenda_index, render_meeting_results, render_agenda_details, render_minutes

state.update(render_node(state))
final = state["final_summary"]

# (1) JSON 확인 (원본 전사문은 길어서 빼고 출력)
print(final.model_dump_json(indent=2, exclude={"meeting_info": {"source_document": {"utterances", "full_text"}}}))


In [ ]:
from node.render import save_final_summary
save_final_summary(state["final_summary"], state["final_markdown"])          # JSON과 txt 함께 저장

In [ ]:
# (2) 블록별 문자열 확인
for render in (render_meeting_info, render_agenda_index, render_meeting_results, render_agenda_details):
    print("\n".join(render(final)), end="\n\n")

In [ ]:
# 안건별 상세의 모든 문장과 근거 발화 원문
texts = {u.utterance_index: u.text for u in final.meeting_info.source_document.utterances}
for d in final.agenda_details:
    groups = [("논의 배경", d.discussion_background), *[(f"주요 논의 {g.number}. {g.subtopic}", g.items) for g in d.major_discussions],
              ("결정사항", d.decisions), ("미해결 이슈", d.unresolved_issues), ("다음 회의 후보 안건", d.next_meeting_topics)]
    for label, items in groups:
        for it in items:
            print(f"[{label}] {it.text}  → 발화 {it.evidence_utterance_indices}")
            for i in it.evidence_utterance_indices:
                print(f"      [{i}] {texts[i]}")
